# EDA and Preprocessing: U.S. Weekly Retail Gasoline Prices

This notebook documents the exploratory data analysis and preprocessing steps used by the forecasting app. It is designed to run from the repository root (`fuel-price-forecast-app/`).

Main questions:

- Is the weekly gasoline price series complete and clean enough for modeling?
- What long-run trend, volatility, and seasonal patterns appear in the data?
- Why do the final model features use lagged prices, month, and ISO week of year?
- How do the EDA observations connect to the XGBoost and SARIMA model choices?

## 1. Imports and Paths

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
RAW_PATH = ROOT / "data" / "raw" / "gasoline_weekly.csv"
PROCESSED_PATH = ROOT / "data" / "processed" / "gasoline_modeling.csv"
METADATA_PATH = ROOT / "models" / "metadata.json"

plt.style.use("seaborn-v0_8-whitegrid")

## 2. Load Raw and Processed Data

The raw file contains the EIA weekly retail gasoline price series. The processed file is produced by `scripts/train.py` after feature engineering.

In [ ]:
raw = pd.read_csv(RAW_PATH, parse_dates=["period"]).sort_values("period")
processed = pd.read_csv(PROCESSED_PATH, parse_dates=["period"]).sort_values("period")
metadata = json.loads(METADATA_PATH.read_text())

display(raw.head())
display(processed.head())
print(f"Raw rows: {len(raw):,}")
print(f"Processed rows: {len(processed):,}")
print(f"Raw date range: {raw['period'].min().date()} to {raw['period'].max().date()}")
print(f"Processed date range: {processed['period'].min().date()} to {processed['period'].max().date()}")

## 3. Data Quality Checks

The modeling pipeline expects one weekly observation per row, no missing gasoline prices, and no duplicate weekly dates.

In [ ]:
quality_summary = pd.DataFrame({
    "check": [
        "missing_period_raw",
        "missing_price_raw",
        "duplicate_period_raw",
        "missing_values_processed",
        "duplicate_period_processed",
    ],
    "value": [
        raw["period"].isna().sum(),
        raw["price"].isna().sum(),
        raw["period"].duplicated().sum(),
        processed.isna().sum().sum(),
        processed["period"].duplicated().sum(),
    ],
})
display(quality_summary)

weekly_gaps = raw["period"].diff().dt.days.value_counts().sort_index()
display(weekly_gaps.rename("count").to_frame())

## 4. Price Distribution and Long-Run Trend

The target variable is weekly dollars per gallon. The series is non-stationary over the long run, with major spikes around energy-market shocks. This supports using lag features and a model that can adapt to recent price levels.

In [ ]:
display(raw["price"].describe().to_frame())

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(raw["period"], raw["price"], color="#2563eb", linewidth=1.5)
ax.set_title("Weekly U.S. retail gasoline price")
ax.set_xlabel("Week")
ax.set_ylabel("$/gal")
plt.show()

## 5. Seasonal Patterns: Month and ISO Week

`month` and `weekofyear` are calendar features. `weekofyear` uses ISO week numbering, so each date gets a week number from 1 to 52 or 53. These features help the model learn recurring seasonal behavior, such as summer driving-season effects.

In [ ]:
seasonal = raw.assign(
    month=raw["period"].dt.month,
    weekofyear=raw["period"].dt.isocalendar().week.astype(int),
)

monthly = seasonal.groupby("month", as_index=False)["price"].mean()
weekly = seasonal.groupby("weekofyear", as_index=False)["price"].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(monthly["month"], monthly["price"], color="#f59e0b")
axes[0].set_title("Average price by month")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Average $/gal")

axes[1].plot(weekly["weekofyear"], weekly["price"], color="#10b981")
axes[1].set_title("Average price by ISO week")
axes[1].set_xlabel("ISO week of year")
axes[1].set_ylabel("Average $/gal")
plt.tight_layout()
plt.show()

## 6. Feature Engineering Checks

The processed dataset uses lag features from 1, 2, 4, 8, and 52 weeks ago. The 52-week lag is useful because it gives the model a same-season comparison from the previous year. The first 52 raw rows are dropped because `lag_52` is not available until week 53.

In [ ]:
feature_cols = metadata["feature_columns"]
print("Feature columns from metadata:", feature_cols)

display(processed[["price"] + feature_cols].corr()[["price"]].sort_values("price", ascending=False))

fig, ax = plt.subplots(figsize=(8, 5))
processed[["price", "lag_1", "lag_4", "lag_52"]].tail(156).plot(ax=ax)
ax.set_title("Recent target vs selected lag features")
ax.set_xlabel("Processed row index")
ax.set_ylabel("$/gal")
plt.show()

## 7. Train/Test Split and Model Evaluation

The app evaluates models on the most recent 52 weeks. This is a time-series holdout, not a random split, because the goal is to forecast future gasoline prices from past observations.

In [ ]:
holdout = 52
train = processed.iloc[:-holdout]
test = processed.iloc[-holdout:]

print(f"Train period: {train['period'].min().date()} to {train['period'].max().date()} ({len(train):,} rows)")
print(f"Holdout period: {test['period'].min().date()} to {test['period'].max().date()} ({len(test):,} rows)")

model_rows = []
for name, details in metadata["models"].items():
    model_rows.append({
        "model": name,
        "MAE ($/gal)": details["validation_mae"],
        "RMSE ($/gal)": details["validation_rmse"],
        "description": details["description"],
    })
model_eval = pd.DataFrame(model_rows).sort_values("MAE ($/gal)")
display(model_eval)

## 8. EDA Conclusions

- The raw EIA series is a clean weekly time series with `period` and `price` columns.
- Gasoline prices show long-run level shifts and sharp shocks, so recent lags are important predictors.
- Month and ISO week features give XGBoost a compact representation of annual seasonality.
- The 52-week holdout is appropriate because the app forecasts future weeks rather than randomly sampled observations.
- XGBoost performs better on the current holdout than SARIMA, so the deployed app uses XGBoost as the default while still allowing SARIMA comparison.